In [ ]:
from pathlib import Path

import pandas as pd

RAW_DIR = Path("raw")

# Load every CSV in raw/ into a dict keyed by file name (without extension)
dfs = {path.stem: pd.read_csv(path) for path in sorted(RAW_DIR.glob("*.csv"))}

for name, df in dfs.items():
    print(f"{name}: {df.shape[0]:,} rows x {df.shape[1]} cols")

In [ ]:
case_management = dfs["case_management_export"]
crm_donations = dfs["crm_donations"]
marketing_daily = dfs["marketing_daily"]
quickbooks_gl = dfs["quickbooks_gl"]
ramp_card_spend = dfs["ramp_card_spend"]

In [ ]:
for name, df in dfs.items():
    print(f"\n=== {name} ===")
    display(df.head())

## Profiling
Row count, column types, blanks, unique values and duplicate rows for every file.

In [ ]:
for name, df in dfs.items():
    print(f"\n=== {name} ===")
    print(f"rows: {len(df):,} | duplicate rows: {df.duplicated().sum():,}")
    display(pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "blanks": df.isna().sum(),
        "blank_%": (df.isna().mean() * 100).round(1),
        "unique": df.nunique(),
    }))

## Value counts on categorical text columns

In [ ]:
TEXT_COLS = {
    "case_management_export": ["county", "status", "outcome"],
    "crm_donations": ["campaign", "source_channel"],
    "marketing_daily": ["platform", "utm_campaign"],
    "quickbooks_gl": ["account", "class"],
    "ramp_card_spend": ["department", "employee", "merchant", "currency"],
}

for name, cols in TEXT_COLS.items():
    for col in cols:
        print(f"\n=== {name}.{col} ===")
        print(dfs[name][col].value_counts(dropna=False).to_string())

## Format checks
Donation amounts, date formats, duplicate IDs, open/closed cases and currencies.

In [ ]:
# Donation amounts are stored as text: find the formats that aren't plain numbers
amount = crm_donations["amount"].dropna().astype(str)
non_numeric = amount[pd.to_numeric(amount, errors="coerce").isna()]
print(f"non-numeric amounts: {len(non_numeric):,}")
print(non_numeric.str.replace(r"\d", "9", regex=True).value_counts().head(10).to_string())

# Strip "$", "," and "USD " to see the numeric range, including negatives
amount_num = pd.to_numeric(amount.str.replace(r"[$,]|USD\s*", "", regex=True), errors="coerce")
print(f"\nunparseable after cleaning: {amount_num.isna().sum()}")
print(f"negative amounts: {(amount_num < 0).sum()}")
amount_num.describe()

In [ ]:
# Date columns: count each format pattern (digits shown as 9, letters as A)
DATE_COLS = {
    "case_management_export": ["date_assigned", "date_closed"],
    "crm_donations": ["donation_date"],
    "marketing_daily": ["date"],
    "quickbooks_gl": ["txn_date"],
    "ramp_card_spend": ["txn_date"],
}

for name, cols in DATE_COLS.items():
    for col in cols:
        patterns = (
            dfs[name][col].dropna().astype(str)
            .str.replace(r"\d", "9", regex=True)
            .str.replace(r"[A-Za-z]", "A", regex=True)
        )
        print(f"\n=== {name}.{col} ===")
        print(patterns.value_counts().to_string())

In [ ]:
# Are repeated IDs only exact duplicate rows, or different rows sharing an ID?
ID_COLS = {
    "case_management_export": "case_id",
    "crm_donations": "donation_id",
    "quickbooks_gl": "qb_txn_id",
    "ramp_card_spend": "ramp_txn_id",
}

for name, id_col in ID_COLS.items():
    df = dfs[name]
    repeated_ids = df[id_col].duplicated().sum()
    exact_dupes = df.duplicated().sum()
    print(f"{name}: repeated {id_col} = {repeated_ids}, exact duplicate rows = {exact_dupes}, "
          f"conflicting = {repeated_ids - exact_dupes}")

In [ ]:
# Do blank date_closed values line up with open cases?
pd.crosstab(
    case_management["status"].str.lower(),
    case_management["date_closed"].isna(),
    rownames=["status"],
    colnames=["date_closed blank"],
)

In [ ]:
# Ramp currency mix
ramp_card_spend.groupby("currency")["amount"].agg(["count", "sum"])